Correlate LMCs and cell type proportions 

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
import seaborn as sns
import matplotlib.pyplot as plt

cell_type_cols = ['epithelial', 'smooth_muscle', 'fibroblast',
                  'endothelial', 'macrophage', 'plasma', 't_cd8',
                  't_cd4', 'b', 'mast', 'monocyte', 'natural_killer',
                  't_regulatory', 'dendritic']
# ── Load both matrices ────────────────────────────────────────────────
# LMC proportions — already aggregated to patient level
lmc_patient = pd.read_csv(
    '/path/to/data/LMC_proportions_per_sample.csv',
    index_col=0
).T
lmc_patient.index = lmc_patient.index.str.extract(r'(PPCG\d+)')[0].values
lmc_patient = lmc_patient.groupby(lmc_patient.index).mean()

# Cell type proportions — already aggregated to patient level
deconv = pd.read_csv(
    '/path/to/data/cell_type_deconvolution_PPCG_TME.csv'
)

deconv['ppcg_id'] = deconv['PPCG_RNA_Assay_ID'].str.extract(r'(PPCG\d+)')[0]
deconv_patient = deconv.groupby('ppcg_id')[cell_type_cols].mean()

# ── Find common patients ──────────────────────────────────────────────
common = lmc_patient.index.intersection(deconv_patient.index)
print(f"Common patients: {len(common)}")

lmc_common = lmc_patient.loc[common]
deconv_common = deconv_patient.loc[common]

print("LMC columns:", lmc_common.columns.tolist())
print("Cell type columns:", deconv_common.columns.tolist())

# ── Compute Spearman correlations: each LMC vs each cell type ─────────
lmc_cols = [f'LMC{i}' for i in range(1, 15)]
cell_type_cols = ['epithelial', 'smooth_muscle', 'fibroblast',
                  'endothelial', 'macrophage', 'plasma', 't_cd8',
                  't_cd4', 'b', 'mast', 'monocyte', 'natural_killer',
                  't_regulatory', 'dendritic']

corr_matrix = pd.DataFrame(index=lmc_cols, columns=cell_type_cols, dtype=float)
pval_matrix = pd.DataFrame(index=lmc_cols, columns=cell_type_cols, dtype=float)

for lmc in lmc_cols:
    for ct in cell_type_cols:
        r, p = spearmanr(lmc_common[lmc], deconv_common[ct])
        corr_matrix.loc[lmc, ct] = r
        pval_matrix.loc[lmc, ct] = p

print("\nSpearman correlation matrix (LMC proportions vs cell type proportions):")
print(corr_matrix.round(3).to_string())

# ── Heatmap ───────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(
    corr_matrix.astype(float),
    cmap='RdBu_r',
    center=0,
    annot=True, fmt='.2f',
    linewidths=0.5,
    ax=ax,
    cbar_kws={'label': 'Spearman r'}
)
ax.set_title(
    'Spearman correlation: LMC proportions vs cell type proportions\n'
    'PPCG cohort (n patients with both methylation and RNA-seq data)',
    fontsize=11, fontweight='bold'
)
ax.set_xlabel('Cell type (RNA-seq deconvolution)')
ax.set_ylabel('LMC (methylation deconvolution)')
plt.tight_layout()
plt.savefig('LMC_celltype_correlation.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved correlation heatmap')

Clustermap:

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

g = sns.clustermap(
    corr_matrix.astype(float),
    cmap='RdBu_r',
    center=0,
    annot=True, fmt='.2f',
    linewidths=0.5,
    figsize=(16, 10),
    dendrogram_ratio=0.15,
    cbar_pos=(1.02, 0.3, 0.03, 0.4),
    cbar_kws={'label': 'Spearman r'}
)
g.ax_heatmap.set_title(
    'Spearman correlation: LMC proportions vs cell type proportions\n'
    'PPCG cohort (n=830 patients with methylation and RNA-seq data)',
    fontsize=11, fontweight='bold'
)
g.ax_heatmap.set_xlabel('Cell type (RNA-seq deconvolution)', fontsize=10)
g.ax_heatmap.set_ylabel('LMC (methylation deconvolution)', fontsize=10)
plt.savefig('LMC_celltype_correlation_clustermap.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved clustermap')

Now GSEA:

In [ ]:
import gseapy as gp
import pandas as pd

# Load ranks file
ranks = pd.read_csv(
    '/path/to/data/LMCs_ranks_prostate.tsv',
    sep='\t'
)

# Read as text and replace /t with actual tab
with open('/path/to/data/LMCs_ranks_prostate.tsv', 'r') as f:
    content = f.read().replace('/t', '\t')

# Parse from the fixed string
from io import StringIO
ranks = pd.read_csv(StringIO(content), sep='\t')

print("Columns:", ranks.columns.tolist())
print("Shape:", ranks.shape)
print("\nHead:")
print(ranks.head())
print("\nLMCs:", ranks['lmc'].unique())

GSEA on three databases:

In [ ]:
import gseapy as gp
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from io import StringIO

# ── Already loaded ranks, now run GSEA per LMC ───────────────────────
lmc_list = sorted(ranks['lmc'].unique())

# ── Run prerank GSEA for each LMC ────────────────────────────────────
# Using Hallmarks gene sets — most interpretable
all_results = {}

for lmc in lmc_list:
    print(f'Running GSEA for {lmc}...')
    
    lmc_ranks = ranks[ranks['lmc'] == lmc][['gene', 'r']].copy()
    lmc_ranks = lmc_ranks.sort_values('r', ascending=False)
    lmc_ranks = lmc_ranks.drop_duplicates('gene')
    
    try:
        res = gp.prerank(
            rnk=lmc_ranks,
            gene_sets=[
                'MSigDB_Hallmark_2020',
                'Reactome_2022',
                'GO_Biological_Process_2023',
                'CellMarker_2024'
            ],
            outdir=None,
            min_size=10,
            max_size=500,
            permutation_num=1000,
            seed=42,
            verbose=False
        )
        all_results[lmc] = res.res2d
        n = len(res.res2d)
        print(f'  {lmc}: {n} gene sets tested')
    except Exception as e:
        print(f'  {lmc}: ERROR — {e}')

print('\nGSEA complete')

# ── Extract significant results ───────────────────────────────────────
sig_results = []
for lmc, df in all_results.items():
    df = df.copy()
    df['lmc'] = lmc
    sig = df[df['FDR q-val'] < 0.25]  # standard GSEA threshold
    sig_results.append(sig)
    print(f'{lmc}: {len(sig)} significant gene sets (FDR<0.25)')

sig_df = pd.concat(sig_results, ignore_index=True)
sig_df.to_csv('GSEA_Hallmarks_per_LMC.csv', index=False)
print(f'\nTotal significant results: {len(sig_df)}')
print(sig_df[['lmc', 'Term', 'NES', 'FDR q-val']].sort_values(['lmc', 'NES']).to_string())

The GSEA ran successfully. 17,843 significant results across all LMCs is a lot — this is because GO Biological Process has thousands of gene sets and FDR<0.25 is a lenient threshold. The next step is to filter and summarise this into something interpretable.
The best approach now is to:

Filter to FDR<0.05 for stricter results
Focus on Hallmarks only for the main figure (clean, 50 gene sets)
Take top 5 positive and top 5 negative NES terms per LMC per database



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load results
sig_df = pd.read_csv(
    '/path/to/data/Cell2location/spatial_mapping/notebooks/GSEA_Hallmarks_per_LMC.csv'
)

print("Columns:", sig_df.columns.tolist())
print("Shape:", sig_df.shape)

# Split database from term name
sig_df['database'] = sig_df['Term'].str.split('__').str[0]
sig_df['term_clean'] = sig_df['Term'].str.split('__').str[1]

print("\nDatabase counts:")
print(sig_df['database'].value_counts())

# ── Focus on Hallmarks first ──────────────────────────────────────────
hallmarks = sig_df[
    (sig_df['database'] == 'MSigDB_Hallmark_2020') &
    (sig_df['FDR q-val'] < 0.05)
].copy()

print(f"\nHallmarks significant (FDR<0.05): {len(hallmarks)}")
print("\nTop results:")
print(hallmarks[['lmc', 'term_clean', 'NES', 'FDR q-val']]
      .sort_values(['lmc', 'NES']).to_string())

# ── Pivot for heatmap ─────────────────────────────────────────────────
pivot_hallmarks = hallmarks.pivot_table(
    index='lmc',
    columns='term_clean',
    values='NES',
    aggfunc='first'
).fillna(0)

print(f"\nHallmarks pivot shape: {pivot_hallmarks.shape}")

# ── Plot heatmap ──────────────────────────────────────────────────────
if pivot_hallmarks.shape[1] > 0:
    fig, ax = plt.subplots(figsize=(20, 8))
    sns.heatmap(
        pivot_hallmarks,
        cmap='RdBu_r',
        center=0,
        linewidths=0.3,
        ax=ax,
        cbar_kws={'label': 'NES (normalised enrichment score)'}
    )
    ax.set_title(
        'GSEA Hallmarks enrichment per LMC\n(FDR < 0.05)',
        fontsize=12, fontweight='bold'
    )
    ax.set_xlabel('Hallmark gene set', fontsize=10)
    ax.set_ylabel('LMC', fontsize=10)
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.tight_layout()
    plt.savefig(
        '/path/to/data/GSEA_Hallmarks_heatmap.png',
        dpi=150, bbox_inches='tight'
    )
    plt.close()
    print('Saved Hallmarks heatmap')

# ── Top 5 positive and negative per LMC for all databases ────────────
top_per_lmc = []
for lmc in sorted(sig_df['lmc'].unique()):
    for db in sig_df['database'].unique():
        subset = sig_df[
            (sig_df['lmc'] == lmc) &
            (sig_df['database'] == db) &
            (sig_df['FDR q-val'] < 0.05)
        ].copy()
        if len(subset) == 0:
            continue
        top_pos = subset.nlargest(5, 'NES')
        top_neg = subset.nsmallest(5, 'NES')
        top_per_lmc.append(pd.concat([top_pos, top_neg]))

top_df = pd.concat(top_per_lmc, ignore_index=True)
top_df[['lmc', 'database', 'term_clean', 'NES', 'FDR q-val']].to_csv(
    '/path/to/data/GSEA_top5_per_LMC.csv',
    index=False
)
print(f'\nSaved top 5 per LMC per database: {len(top_df)} rows')
print('\nTop terms per LMC (Hallmarks only):')
print(top_df[top_df['database'] == 'MSigDB_Hallmark_2020'][
    ['lmc', 'term_clean', 'NES', 'FDR q-val']
].sort_values(['lmc', 'NES']).to_string())

Plot clustermap instead of heatmap:

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load results
sig_df = pd.read_csv(
    '/path/to/data/Cell2location/spatial_mapping/notebooks/GSEA_Hallmarks_per_LMC.csv'
)

print("Columns:", sig_df.columns.tolist())
print("Shape:", sig_df.shape)

# Split database from term name
sig_df['database'] = sig_df['Term'].str.split('__').str[0]
sig_df['term_clean'] = sig_df['Term'].str.split('__').str[1]

print("\nDatabase counts:")
print(sig_df['database'].value_counts())

# ── Focus on Hallmarks first ──────────────────────────────────────────
hallmarks = sig_df[
    (sig_df['database'] == 'MSigDB_Hallmark_2020') &
    (sig_df['FDR q-val'] < 0.05)
].copy()

print(f"\nHallmarks significant (FDR<0.05): {len(hallmarks)}")
print("\nTop results:")
print(hallmarks[['lmc', 'term_clean', 'NES', 'FDR q-val']]
      .sort_values(['lmc', 'NES']).to_string())

# ── Pivot for heatmap ─────────────────────────────────────────────────
pivot_hallmarks = hallmarks.pivot_table(
    index='lmc',
    columns='term_clean',
    values='NES',
    aggfunc='first'
).fillna(0)

print(f"\nHallmarks pivot shape: {pivot_hallmarks.shape}")

In [ ]:
# ── Plot clustermap (with row and column dendrograms) ──────────────────
if pivot_hallmarks.shape[1] > 0:
    # sns.clustermap creates its own figure and axes grids automatically.
    # 'metric' and 'method' define how the hierarchical clustering calculates distance.
    g = sns.clustermap(
        pivot_hallmarks,
        method='ward',                # Ward linkage minimizes variance within clusters
        metric='euclidean',           # Standard geometric distance in NES space
        cmap='RdBu_r',
        center=0,
        linewidths=0.3,
        figsize=(20, 10),             # Adjusted height slightly to give dendrograms room
        cbar_kws={'label': 'NES (normalised enrichment score)'}
    )
    
    # Adjust titles and labels on the internally managed axis grids
    g.fig.suptitle(
        'GSEA Hallmarks enrichment per LMC\nHierarchical Clustering (FDR < 0.05)',
        fontsize=14, fontweight='bold', y=1.02
    )
    
    # Access the underlying heatmap axes to set titles
    g.ax_heatmap.set_xlabel('Hallmark gene set', fontsize=11)
    g.ax_heatmap.set_ylabel('LMC', fontsize=11)
    
    # Rotate the column labels on the cluster axis grid so they don't overlap
    plt.setp(g.ax_heatmap.get_xticklabels(), rotation=45, ha='right', fontsize=9)
    plt.setp(g.ax_heatmap.get_yticklabels(), rotation=0, fontsize=10)
    
    # Save the figure using the cluster map figure handler
    g.savefig(
        '/path/to/data/GSEA_Hallmarks_clustermap.png',
        dpi=150, bbox_inches='tight'
    )
    plt.close()
    print('Saved Hallmarks clustermap with dendrograms')

plot for each database: 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load results
sig_df = pd.read_csv(
    '/path/to/data/Cell2location/spatial_mapping/notebooks/GSEA_Hallmarks_per_LMC.csv'
)

# Split database from term name
sig_df['database'] = sig_df['Term'].str.split('__').str[0]
sig_df['term_clean'] = sig_df['Term'].str.split('__').str[1]

print("Database counts:")
print(sig_df['database'].value_counts())

# ── Plot clustermap for each database ────────────────────────────────
databases = {
    'MSigDB_Hallmark_2020':        'GSEA_Hallmarks_clustermap.png',
    'Reactome_2022':               'GSEA_Reactome_clustermap.png',
    'GO_Biological_Process_2023':  'GSEA_GO_BP_clustermap.png',
    'CellMarker_2024':             'GSEA_CellMarker_clustermap.png',
}

for db, filename in databases.items():
    subset = sig_df[
        (sig_df['database'] == db) &
        (sig_df['FDR q-val'] < 0.05)
    ].copy()

    print(f"\n{db}: {len(subset)} significant terms (FDR<0.05)")

    if len(subset) == 0:
        print(f"  Skipping — no significant terms")
        continue

    pivot = subset.pivot_table(
        index='lmc',
        columns='term_clean',
        values='NES',
        aggfunc='first'
    ).fillna(0)

    print(f"  Pivot shape: {pivot.shape}")

    # For GO and Reactome which have many terms, take top N by variance
    # to keep the figure readable
    max_terms = 60
    if pivot.shape[1] > max_terms:
        top_cols = pivot.var(axis=0).nlargest(max_terms).index
        pivot = pivot[top_cols]
        print(f"  Trimmed to top {max_terms} terms by variance")

    # Scale figure width to number of terms
    fig_width = max(16, min(pivot.shape[1] * 0.35, 40))
    fig_height = max(8, pivot.shape[0] * 0.6)

    g = sns.clustermap(
        pivot,
        method='ward',
        metric='euclidean',
        cmap='RdBu_r',
        center=0,
        linewidths=0.3,
        figsize=(fig_width, fig_height),
        cbar_kws={'label': 'NES'}
    )

    g.fig.suptitle(
        f'GSEA {db} enrichment per LMC\n(FDR < 0.05, top {max_terms} terms by variance)',
        fontsize=12, fontweight='bold', y=1.02
    )
    g.ax_heatmap.set_xlabel('Gene set', fontsize=10)
    g.ax_heatmap.set_ylabel('LMC', fontsize=10)
    plt.setp(g.ax_heatmap.get_xticklabels(), rotation=45, ha='right', fontsize=7)
    plt.setp(g.ax_heatmap.get_yticklabels(), rotation=0, fontsize=10)

    outpath = f'/path/to/data/{filename}'
    g.savefig(outpath, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'  Saved {outpath}')

Get the top significant per database:

In [ ]:
import pandas as pd

sig_df = pd.read_csv(
    '/path/to/data/Cell2location/spatial_mapping/notebooks/GSEA_Hallmarks_per_LMC.csv'
)

sig_df['database'] = sig_df['Term'].str.split('__').str[0]
sig_df['term_clean'] = sig_df['Term'].str.split('__').str[1]

print("Database counts (all results):")
print(sig_df['database'].value_counts())

# ── Top 3 positive and negative NES per LMC for each database ─────────
for db in ['MSigDB_Hallmark_2020', 'Reactome_2022', 
           'GO_Biological_Process_2023', 'CellMarker_2024']:
    
    subset_db = sig_df[
        (sig_df['database'] == db) &
        (sig_df['FDR q-val'] < 0.05)
    ].copy()
    
    print(f"\n{'='*60}")
    print(f"=== {db} — FDR<0.05: {len(subset_db)} significant terms ===")
    print(f"{'='*60}")
    
    if len(subset_db) == 0:
        print("  No significant terms at FDR<0.05")
        continue
    
    for lmc in sorted(subset_db['lmc'].unique()):
        lmc_subset = subset_db[subset_db['lmc'] == lmc]
        top_pos = lmc_subset.nlargest(3, 'NES')[['term_clean', 'NES', 'FDR q-val']]
        top_neg = lmc_subset.nsmallest(3, 'NES')[['term_clean', 'NES', 'FDR q-val']]
        print(f"\n  {lmc} ({len(lmc_subset)} sig terms):")
        print("    Top positive NES:")
        for _, row in top_pos.iterrows():
            print(f"      {row['term_clean'][:60]:<60} NES={row['NES']:.3f} FDR={row['FDR q-val']:.4f}")
        print("    Top negative NES:")
        for _, row in top_neg.iterrows():
            print(f"      {row['term_clean'][:60]:<60} NES={row['NES']:.3f} FDR={row['FDR q-val']:.4f}")